In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [3]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

def llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [4]:
## Before RAG

question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

You're interested in joining a course. However, I'm a large language model, I don't have real-time access to specific course information or enrollment deadlines. Can you please provide more details about the course you're interested in, such as the name, provider, or topic? This will help me better understand your query.


In [49]:

# After RAG
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit 
your project while we're still accepting submissions.
"""

prompt = f"""
Your task is to answer questions from course participants
based on the provided context.

If the answer is not in the context, say "I don't know."

Question: {question}

Context: {context}
"""

answer = llm(prompt)
print(answer)

Yes, you can still join the course.


In [6]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [7]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1349

In [8]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [9]:
# Build the search index
import minsearch

index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)
print(f"Total documents: {len(documents)}")

Total documents: 1349


In [10]:
index.fit(documents)
print("Index ready ")

Index ready 


In [11]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5
    )

    return results

In [12]:
results = search("I just discovered the course. Can I join now?")

for r in results:
    print(r['question'])
    print(r['course'])
    print()

Course: Can I still join the course after the start date?
data-engineering-zoomcamp

Course: What can I do before the course starts?
data-engineering-zoomcamp

Course: When does the course start?
data-engineering-zoomcamp

Course - Can I follow the course after it finishes?
data-engineering-zoomcamp

How can we contribute to the course?
data-engineering-zoomcamp



In [13]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [14]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [15]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [16]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
If the CONTEXT doesn't contain the answer, output NONE.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

    context = ""
    for doc in search_results:
        context += f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['answer']}\n\n"

    return prompt_template.format(question=query, context=context)

In [17]:
question = "I just discovered the course. Can I join now?"
results = search(question)
prompt = build_prompt(question, results)
print(prompt)

You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
If the CONTEXT doesn't contain the answer, output NONE.

QUESTION: I just discovered the course. Can I join now?

CONTEXT:
section: General Course-Related Questions
question: Course: Can I still join the course after the start date?
answer: Yes, even if you don't register, you're still eligible to submit the homework.

Be aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.

section: General Course-Related Questions
question: Course: What can I do before the course starts?
answer: Get the basic environment ready ahead of time:

- Google Cloud account (free trial — see the GCP setup FAQ).
- Google Cloud SDK (`gcloud` CLI).
- Python 3 — install via your OS package manager or [`uv`](https://docs.astral.sh/uv/) (recommended for managing Pyth

In [18]:
def rag(question):
    search_results = search(question)
    prompt = build_prompt(question, search_results)
    answer = llm(prompt)
    return answer

In [19]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

You can still join the course after the start date, and even if you don't register, you're still eligible to submit the homework.


In [20]:
rag("How do I get a certificate?")

'To get a certificate, follow these steps:\n\n1. Verify your name is correct on the certificate by editing it in your course profile under "Edit Course Profile" if not.\n2. Wait for the second announcement in the course Telegram and Slack channels stating that grading is complete and the certificate is ready.\n3. Log into your enrollment page on the cohort\'s course site (`https://courses.datatalks.club/de-zoomcamp-<year>/enrollment`) and follow the instructions in [certificates.md](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/certificates.md) to generate the certificate document.\n\nNote that you do not need to complete the homeworks to get the certificate, as long as you complete the peer-reviewed capstone projects on time. However, you must finish the course with a “live” cohort, and you cannot receive a certificate for the self-paced mode.'

In [21]:
rag("How do i unenroll?")

'NONE'

In [22]:
rag("who is the creator/creators of this course")

'NONE\n\nThere is no explicit information about the creator/creators of this course in the provided CONTEXT.'

In [23]:
rag("what benefit will i accrue from learning this course")

'Based on the provided context, the answer to your question "what benefit will I accrue from learning this course" seems to be indirectly mentioned under "Course: How many hours per week am I expected to spend on this course?" in the context of preparing yourself for the course.\n\nHowever, the context does not explicitly give you the direct benefit you\'re asking for.\n\nThe answer can infer that learning the course will enable you to spend 5-15 hours a week and prepare you for the course ahead of time. However the main benefit is actually the benefit of learning how to work in data engineering, which in the FAQs is not given.\n\nIf we interpret "what benefit will I accrue from learning this course" to be what can you do after learning this course. Then it is actually given in "Course: What can I do before the course starts?" under section: General Course-Related Questions as:\n\n- Google Cloud account (free trial — see the GCP setup FAQ).\n- Google Cloud SDK (`gcloud` CLI).\n- Python

In [51]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, even if you don't register, you're still eligible to submit the homework, but be aware that there will be deadlines for turning in homeworks and the final projects.
